# RT-LCOD — Run All local RTX A3000: dữ liệu thật → train → distill → test → video UI

Notebook này bám đúng kiến trúc đã chốt:

**YOLOE-26M (open-vocabulary) + lightweight Attribute/Relation Grounding Head + NO_TARGET + Grounding DINO teacher chỉ dùng offline.**

Mục tiêu của notebook là **bấm Run All một lần**. Nó tự chuẩn bị một tập gRefCOCO thực, cache proposal/feature, train Stage 1, chạy teacher distillation Stage 2, chọn checkpoint trên validation, test held-out, nạp runtime cuối và hiển thị UI video có ô nhập prompt.

Mọi session đều in log: môi trường/GPU, dữ liệu, proposal recall, loss/accuracy, checkpoint, thời gian, VRAM và latency. Giữ toàn bộ output của notebook để dùng cho vòng tối ưu tiếp theo.


## Session 0 — Cấu hình một chỗ duy nhất

Các giá trị mặc định được chọn cho RTX A3000 Laptop GPU 12 GB. Sau khi có log thật mới tăng dataset/epochs hoặc đổi input size.


In [ ]:
# ===== USER CONTROLS — local RTX A3000 Laptop GPU 12 GB =====
RUN_PROFILE = "local_a3000"
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 150
TEST_SAMPLES = 150
TEACHER_MAX_SAMPLES = 300     # Teacher offline; giữ nhỏ để ổn định trên 12 GB VRAM
DATA_DOWNLOAD_WORKERS = 4
PREFER_HTTP_COCO = True       # Mạng hiện tại lỗi TLS với CDN COCO; chỉ dùng HTTP công khai cho ảnh COCO
RUN_TEACHER = True            # False = bỏ Stage 2 để chạy nhanh hơn
REBUILD_CANDIDATE_CACHE = False

# test.mp4 ở thư mục gốc của project (ngoài rt_lcod).
VIDEO_PATH = "../test.mp4"
DEFAULT_PROMPT = "person"

print("[CONFIG] profile =", RUN_PROFILE)
print("[CONFIG] train/val/test =", TRAIN_SAMPLES, VAL_SAMPLES, TEST_SAMPLES)
print("[CONFIG] teacher_max_samples =", TEACHER_MAX_SAMPLES, "run_teacher =", RUN_TEACHER)
print("[CONFIG] data_download_workers =", DATA_DOWNLOAD_WORKERS)
print("[CONFIG] prefer_http_coco =", PREFER_HTTP_COCO)
print("[CONFIG] rebuild_candidate_cache =", REBUILD_CANDIDATE_CACHE)
print("[CONFIG] default video =", VIDEO_PATH)
print("[CONFIG] default prompt =", DEFAULT_PROMPT)


## Session 1 — Kiểm tra project local, runtime và tạo RunAll orchestrator

Cell này tự tìm project local và kiểm tra Python 3.14, CUDA cùng các thư viện đã cài sẵn. Notebook **không clone, không cập nhật Git và không tự cài thư viện**. Cuối cell tạo `runner`, là đối tượng điều phối toàn bộ pipeline.


In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()

def find_root() -> Path | None:
    candidates = [HERE, HERE / "rt_lcod", HERE.parent, HERE.parent / "rt_lcod"]
    return next((path for path in candidates if (path / "pyproject.toml").is_file()), None)

ROOT = find_root()
if ROOT is None:
    raise RuntimeError("Không tìm thấy rt_lcod. Mở notebook từ thư mục training_test_model hoặc rt_lcod.")

try:
    import cv2, torch, yaml  # noqa: F401
except ImportError as exc:
    raise RuntimeError("Thiếu thư viện bắt buộc trong Python 3.14 (LCOD).") from exc

if not torch.cuda.is_available():
    raise RuntimeError("CUDA PyTorch không khả dụng. Hãy mở Jupyter bằng Python 3.14 (LCOD).")

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("[SETUP] ROOT =", ROOT)
print("[SETUP] Python =", sys.version.replace("\n", " "))
print("[SETUP] GPU =", torch.cuda.get_device_name(0))

from rt_lcod.runall import RunAll

runner = RunAll(
    ROOT,
    train_samples=TRAIN_SAMPLES,
    val_samples=VAL_SAMPLES,
    test_samples=TEST_SAMPLES,
    teacher_max_samples=TEACHER_MAX_SAMPLES,
    run_teacher=RUN_TEACHER,
    rebuild_candidate_cache=REBUILD_CANDIDATE_CACHE,
    config_name=RUN_PROFILE,
    data_download_workers=DATA_DOWNLOAD_WORKERS,
    prefer_http_coco=PREFER_HTTP_COCO,
)
print("[SETUP] RunAll ready")


## Session 2 — RTX A3000 / CUDA preflight

Dừng ngay nếu CUDA không khả dụng. Log này cho biết GPU, VRAM, CUDA/PyTorch và config inference/training thực tế.


In [ ]:
runner.preflight()


## Session 3 — Dataset thật: gRefCOCO → canonical train/val/test

Tự tải metadata gRefCOCO và **chỉ các COCO image cần cho subset hiện tại**. V1 giữ single-target hoặc true no-target samples. Dataset được tách theo official split, sau đó validate schema và in thống kê/samples.


In [ ]:
runner.prepare_data()


## Session 4 — YOLOE proposal + frozen visual/text cache

Mỗi sample chạy YOLOE-26M-seg một lần để lấy candidate. Sau đó MobileNetV3-Small + ROIAlign lấy region features và MiniLM lấy attribute/relation embedding. Cache giúp 10 epoch training không phải chạy lại YOLOE.

Quan trọng nhất ở đây là `positive_proposal_recall_at_iou`: nếu YOLOE không propose target thì grounding head không thể cứu.


In [ ]:
runner.cache_candidates()


## Session 5 — Stage 1: train grounding student trên dữ liệu thật

YOLOE, MobileNet và MiniLM vẫn frozen. Chỉ train lightweight grounding head: visual projection, attribute score, relation score, candidate fusion và NO_TARGET. Trainer lưu `last.pt`, `best.pt`, checkpoint mỗi epoch, metrics và timing.

Sau train, validation chạy ngay và in metric.


In [ ]:
runner.train_stage1()
stage1_val = runner.evaluate_stage1_validation()
print("[NOTEBOOK] Stage 1 validation:")
print(stage1_val)


## Session 6 — Grounding DINO teacher cache (offline)

Teacher **không chạy khi deploy video**. Nó chỉ tạo soft distribution trên một subset train để student học thêm cross-modal grounding. Để phù hợp RTX A3000 12 GB, mặc định teacher dùng tối đa `TEACHER_MAX_SAMPLES`.


In [ ]:
runner.cache_teacher_outputs()


## Session 7 — Stage 2: Knowledge Distillation

Khởi tạo student từ `best.pt` Stage 1 nhưng bắt đầu optimizer/scheduler mới. Các sample có teacher target dùng KL distillation; sample chưa có teacher vẫn dùng supervised losses bình thường.


In [ ]:
runner.train_stage2()


## Session 8 — Chọn model cuối bằng VALIDATION, không nhìn TEST

So Stage 1 và Stage 2 trên validation bằng composite `0.70 * target_top1 + 0.30 * no_target_accuracy`. Checkpoint tốt hơn được chọn làm final model.


In [ ]:
FINAL_CHECKPOINT = runner.select_final_on_validation()
print("[NOTEBOOK] FINAL_CHECKPOINT =", FINAL_CHECKPOINT)


## Session 9 — Held-out TEST một lần

Chỉ sau khi model đã được chọn trên validation mới đánh giá test. Log này là kết quả khách quan đầu tiên để so sánh các lần tối ưu sau.


In [ ]:
final_test = runner.final_test()
print("[NOTEBOOK] FINAL TEST:")
print(final_test)


## Session 10 — Load runtime cuối

Nạp checkpoint đã chọn + YOLOE-26M + frozen region/text encoders lên RTX A3000. Sau cell này model đã sẵn sàng cho video.


In [ ]:
runtime = runner.load_runtime()


## Session 11 — Video player + ô prompt

Đây là cell cuối của **Run All**. UI xuất hiện ngay cả khi `VIDEO_PATH` mặc định chưa tồn tại.

1. Nhập đường dẫn video vào ô **Video** (ví dụ `../test.mp4`).
2. Nhập prompt V1, ví dụ `red car`, `person next to bicycle`, `bottle left of laptop`.
3. Bấm **Play**.
4. Có thể đổi prompt trong lúc video đang chạy.
5. `Infer every = 1` là raw per-frame model; tăng lên 2–5 là chế độ thực dụng nhanh hơn.

Log bên dưới video ghi detector/region/head latency, candidate count, confidence, VRAM và cuối video có mean/p50/p95/p99/FPS.


In [ ]:
player = runner.show_video_ui(VIDEO_PATH, DEFAULT_PROMPT)
print("[NOTEBOOK] Run All completed. The interactive video UI above is ready.")
